# online retail II -  Data Exploration

# Why python instead of excel 
excel has a limit of about 1.05 million rows per sheet but the dataset contains approximately 1.07 million rows which makes it hard to read all the rows and that's why excel ignored the rows more that 10million that's why we switched to pandas for proper analysis at scale.

# Goals for this notebook
1. load the dataset
2. understand structure, size and quality of dataset
3. Identify issues with the data to address before modelling

In [1]:
import pandas as pd

#load the data
df = pd.read_csv('online_retail_II.csv', parse_dates=['InvoiceDate'])
print(df.shape)
df.head(n=50)

(1067371, 8)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom
5,489434,22064,PINK DOUGHNUT TRINKET POT,24,2009-12-01 07:45:00,1.65,13085.0,United Kingdom
6,489434,21871,SAVE THE PLANET MUG,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom
7,489434,21523,FANCY FONT HOME SWEET HOME DOORMAT,10,2009-12-01 07:45:00,5.95,13085.0,United Kingdom
8,489435,22350,CAT BOWL,12,2009-12-01 07:46:00,2.55,13085.0,United Kingdom
9,489435,22349,"DOG BOWL , CHASING BALL DESIGN",12,2009-12-01 07:46:00,3.75,13085.0,United Kingdom


# Initial observations from first 10 rows
1. stockcode has alphanumeric values, means some are pure numbers(85048,22041, etc..), others have letters (79323P, 79323W, etc..) which is normal as stockcodes are identifiers not numbers, so mixing digits is fine. The last letters here indicates color of the products like P for pink or W for white.
2. customer IDs are float numbers (13085.0 instead of 13085), this happens because pandas loaded the columns as float64 instead of int64. This is because there might be some missing values from customer IDs column and pandas can't store NaN values in an integer column, so it converts the whole column to float

## 

In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1067371 non-null  object        
 1   StockCode    1067371 non-null  object        
 2   Description  1062989 non-null  object        
 3   Quantity     1067371 non-null  int64         
 4   InvoiceDate  1067371 non-null  datetime64[ns]
 5   Price        1067371 non-null  float64       
 6   Customer ID  824364 non-null   float64       
 7   Country      1067371 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 65.1+ MB


In [3]:
df.describe()

,Quantity,InvoiceDate,Price,Customer ID
count,1.067371e+06,1067371,1.067371e+06,824364.000000
mean,9.938898e+00,2011-01-02 21:13:55.394028544,4.649388e+00,15324.638504
min,-8.099500e+04,2009-12-01 07:45:00,-5.359436e+04,12346.000000
25%,1.000000e+00,2010-07-09 09:46:00,1.250000e+00,13975.000000
50%,3.000000e+00,2010-12-07 15:28:00,2.100000e+00,15255.000000
75%,1.000000e+01,2011-07-22 10:23:00,4.150000e+00,16797.000000
max,8.099500e+04,2011-12-09 12:50:00,3.897000e+04,18287.000000
std,1.727058e+02,NaN,1.235531e+02,1697.464450


## Summary of statistical findings

# Quantity
1. Range : -80995 to 80995
2. median : 3 items per order (most orders are small)
3. Mean : roughly 9.94
4. std : 172.70 (huge variation, which indicates bulk orders or data error)

# Price
1. Range : -£53594 to £38970 pounds
2. Median : £2.10 (typical item is cheap)
3. Mean :  £4.65
4. std : £123.5 (again high variation, might be because most items are cheap but some are very expensive or represents bulk total)
5. Max : £38970 (which is very high, this looks suspicious and need to check if its per unit price or total of bulk order)

# Date range
1. 1 Dec 2009 to 9 Dec 2011 (~ 2 years)
2. Mean :  2 Jan 2011 (suggests that the data is normally distributed over time because this date is a midpoint)
3. std :  standard deviation doesn't apply to dates, so pandas shows this as NaN

## what to check before moving to data cleaning
1. Max quantity :  80995 (is it actually bulk order or error?)
2. Negative prices and quantities (check for returns/cancellations, need handling strategy)



In [4]:
df[df['Quantity'] == 80995]



,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
1065882,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2011-12-09 09:15:00,2.08,16446.0,United Kingdom


In [5]:
df[df['Quantity'] == -80995] 


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
1065883,C581484,23843,"PAPER CRAFT , LITTLE BIRDIE",-80995,2011-12-09 09:27:00,2.08,16446.0,United Kingdom


In [6]:
df.sort_values(by='Quantity', ascending=False).head(10)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
1065882,581483,23843,"PAPER CRAFT , LITTLE BIRDIE",80995,2011-12-09 09:15:00,2.08,16446.0,United Kingdom
587080,541431,23166,MEDIUM CERAMIC TOP STORAGE JAR,74215,2011-01-18 10:01:00,1.04,12346.0,United Kingdom
90857,497946,37410,BLACK AND WHITE PAISLEY FLOWER MUG,19152,2010-02-15 11:57:00,0.10,13902.0,Denmark
127168,501534,21091,SET/6 WOODLAND PAPER PLATES,12960,2010-03-17 13:09:00,0.10,13902.0,Denmark
127166,501534,21099,SET/6 STRAWBERRY PAPER CUPS,12960,2010-03-17 13:09:00,0.10,13902.0,Denmark
127169,501534,21085,SET/6 WOODLAND PAPER CUPS,12744,2010-03-17 13:09:00,0.10,13902.0,Denmark
1027583,578841,84826,ASSTD DESIGN 3D PAPER STICKERS,12540,2011-11-25 15:57:00,0.00,13256.0,United Kingdom
127167,501534,21092,SET/6 STRAWBERRY PAPER PLATES,12480,2010-03-17 13:09:00,0.10,13902.0,Denmark
192197,507637,84016,FLAG OF ST GEORGE CAR FLAG,10200,2010-05-10 14:55:00,0.00,NaN,United Kingdom
135028,502269,21982,PACK OF 12 SUKI TISSUES,10000,2010-03-23 15:36:00,0.25,17940.0,United Kingdom


In [7]:
invoice_cancellations = (df['Invoice'].str.startswith('C')).sum()
print(invoice_cancellations)

19494


In [8]:
total_rows = len(df)
cancellation_percentage = (invoice_cancellations / total_rows) * 100
print(cancellation_percentage)

1.8263565339511754


In [9]:
df['quantity_range'] = pd.cut(df['Quantity'], bins=[-100000,-10000, -1000, -100, -10,-1,0,1,10,100,1000,10000,100000])
df['is_cancellation'] = df['Invoice'].str.startswith('C')
cancellation_by_quantity = df.groupby('quantity_range')['is_cancellation'].agg(['sum', 'count', 'mean'])
cancellation_by_quantity['cancellation_pct'] = cancellation_by_quantity['mean'] * 100
print(cancellation_by_quantity)

                     sum   count      mean  cancellation_pct
quantity_range                                              
(-100000, -10000]      2       2  1.000000        100.000000
(-10000, -1000]       41     161  0.254658         25.465839
(-1000, -100]        434    1075  0.403721         40.372093
(-100, -10]         3203    4924  0.650487         65.048741
(-10, -1]          15813   16788  0.941923         94.192280
(-1, 0]                0       0       NaN               NaN
(0, 1]                 1  294346  0.000003          0.000340
(1, 10]                0  489170  0.000000          0.000000
(10, 100]              0  250344  0.000000          0.000000
(100, 1000]            0   10209  0.000000          0.000000
(1000, 10000]          0     343  0.000000          0.000000
(10000, 100000]        0       9  0.000000          0.000000


/var/folders/ls/f5wyvk955bnckhvw2n65ydbm0000gn/T/ipykernel_42079/1631735182.py:3: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  cancellation_by_quantity = df.groupby('quantity_range')['is_cancellation'].agg(['sum', 'count', 'mean'])


In [10]:
# How many rows have negative quantity?
negative_qty = (df['Quantity'] < 0).sum()
print(f"Rows with negative quantity: {negative_qty:,}")

# How many rows have 'C' prefix?
c_invoices = df['is_cancellation'].sum()
print(f"Rows with 'C' invoice: {c_invoices:,}")

# How many have BOTH negative quantity AND 'C' prefix?
both = ((df['Quantity'] < 0) & (df['is_cancellation'])).sum()
print(f"Rows with BOTH negative qty and 'C' prefix: {both:,}")

# How many have negative quantity but NO 'C' prefix?
negative_no_c = ((df['Quantity'] < 0) & (~df['is_cancellation'])).sum()
print(f"Negative qty WITHOUT 'C' prefix: {negative_no_c:,}")

# How many have 'C' prefix but POSITIVE quantity?
c_but_positive = ((df['Quantity'] > 0) & (df['is_cancellation'])).sum()
print(f"'C' invoice but POSITIVE quantity: {c_but_positive:,}")

Rows with negative quantity: 22,950
Rows with 'C' invoice: 19,494
Rows with BOTH negative qty and 'C' prefix: 19,493
Negative qty WITHOUT 'C' prefix: 3,457
'C' invoice but POSITIVE quantity: 1


## Identifying returns in the dataset

**Research:**
In retail systems, returns can be recorded as:
- **Financial perspective:** Negative quantity (reverses revenue)
- **Inventory perspective:** Positive quantity (items back in stock)
- **Hybrid systems:** Separate transactions for financial vs. inventory impact

**References:** 
    https://help.shopline.com/hc/en-001/articles/25421340786073-Sales-Report-Understanding-How-Sales-Data-Works

## Findings in the dataset:
Returns use TWO methods, both with negative quantities:
1. 'C' invoice prefix + negative quantity (19,493 rows), where the 'C' prefix indicates a cancellation/return or credit note (which will be investigated later).
2. Regular invoice + negative quantity (3,457 rows)

**Evidence:**
- 22,950 total rows with negative quantity
- out of 22950 rows, 19,494 total rows with 'C' prefix in invoices
- out of 19494 rows with 'C' prefix, 19,493 'C' invoices have negative quantity (99.995%)
- Only 1 'C' invoice has positive quantity (likely data error)

**Conclusion:**
This dataset uses the financial accounting convention — returns are recorded as negative quantities to reverse revenue. The 'C' prefix is a credit note flag that almost always accompanies negative quantities, but 15% of returns (3,457 rows) use regular invoice numbers.

**Decision for analysis:**
Define returns as: `Quantity < 0` (captures all 22,950 return transactions)

The single 'C' invoice with positive quantity will be investigated separately as a potential data quality issue.



In [11]:
df['revenue'] = df['Quantity'] * df['Price']
c_invoice_revenue = df[df['is_cancellation']]['revenue'].sum()
regular_invoice_revenue = df[~df['is_cancellation']]['revenue'].sum()
print(f"'c' invoice total revenue: £{c_invoice_revenue:,.2f}")
print(f"Regular invoice total revenue: £{regular_invoice_revenue:,.2f}")

c_negative_revenue = (df[df['is_cancellation']]['revenue']<0).sum()
C_total_revenue = df['is_cancellation'].sum()
print(f"\n'c' invoice with negative revenue: {c_negative_revenue:,} / {C_total_revenue:,} ({c_negative_revenue/C_total_revenue*100:.2f}%)")

'c' invoice total revenue: £-1,526,667.86
Regular invoice total revenue: £20,813,918.43

'c' invoice with negative revenue: 19,493 / 19,494 (99.99%)


## Credit note verification

**Initial assumption:** The 'C' prefix in invoice numbers indicates credit notes (financial reversals of prior sales).

**Verification approach:** Calculate the revenue impact of 'C' invoices to confirm they reduce net revenue.

**Findings:**
1. **Revenue impact:** 'C' invoices contribute £-1,526,667.86 (negative revenue)
2. **Consistency:** 19,493 / 19,494 'C' invoices (99.99%) have negative revenue
3. **Magnitude:** Credits represent 7.3% of gross sales (£1.5M credits vs £20.8M sales)

**Conclusion:** Confirmed — 'C' prefix indicates credit notes.

In [12]:
# Find the one 'C' invoice with positive quantity
anomaly = df[(df['is_cancellation']) & (df['Quantity'] > 0)]
print(anomaly)

       Invoice StockCode Description  Quantity         InvoiceDate   Price  \
76799  C496350         M      Manual         1 2010-02-01 08:24:00  373.57   

       Customer ID         Country quantity_range  is_cancellation  revenue  
76799          NaN  United Kingdom         (0, 1]             True   373.57  


## Investigating: 'C' invoice with positive quantity

**Finding:** The single 'C' invoice with positive revenue (Invoice C496350) is a manual adjustment entry, not a true credit note error.

**Details:**
- StockCode: "M" (Manual adjustment)
- Description: "Manual"
- Quantity: +1, Price: £373.57, Revenue: +£373.57
- Customer ID: missing (admin entry, not customer transaction)
- Date: 2010-02-01

**Interpretation:**
This is an administrative correction processed through the credit invoice system. Despite the 'C' prefix, it represents a positive financial adjustment (likely reversing a previous over-credit or recording a non-product charge).

**Implication:** The dataset contains non-product entries (e.g., StockCode "M") that aren't real sales or returns. Need to identify all special StockCodes for proper data cleaning.

In [13]:
# Find StockCodes that aren't standard product codes
# Standard codes are typically 5-digit numbers, sometimes with letter suffixes
non_standard = df[df['StockCode'].str.len() < 5]
print("StockCodes shorter than 5 characters:")
print(non_standard['StockCode'].value_counts())

StockCodes shorter than 5 characters:
StockCode
POST    2122
DOT     1446
M       1421
C2       282
D        177
S        104
PADS      19
CRUK      16
B          6
m          5
GIFT       1
C3         1
Name: count, dtype: int64


In [14]:
print(df.nunique())

Invoice            53628
StockCode           5305
Description         5698
Quantity            1057
InvoiceDate        47635
Price               2807
Customer ID         5942
Country               43
quantity_range        11
is_cancellation        2
revenue             9113
dtype: int64


In [15]:
df['Country'].value_counts().head(10)

Country
United Kingdom    981330
EIRE               17866
Germany            17624
France             14330
Netherlands         5140
Spain               3811
Switzerland         3189
Belgium             3123
Portugal            2620
Australia           1913
Name: count, dtype: int64

In [16]:
earliest = df['InvoiceDate'].min()
latest = df['InvoiceDate'].max()
span = latest - earliest

print(f"Earliest transaction: {earliest}")
print(f"Latest transaction: {latest}")
print(f"Time span: {span}")

Earliest transaction: 2009-12-01 07:45:00
Latest transaction: 2011-12-09 12:50:00
Time span: 738 days 05:05:00


# Data Exploration — Summary

## Dataset overview
- **Source:** UCI Machine Learning Repository (via Kaggle: mashlyn/online-retail-ii-uci)
- **Size:** 1,067,371 rows × 8 original columns
- **Period:** 1 Dec 2009 – 9 Dec 2011 (738 days, ~2 years)
- **Geography:** 43 countries, dominated by UK (~92% of transactions)
- **Scale:** 53,628 unique invoices, 5,942 unique customers, 5,305 unique StockCodes
- **Gross revenue:** £20,813,918 (regular invoices)
- **Credits/returns:** £-1,526,668 (7.3% of gross)
- **Net revenue:** £19,287,250

## Data quality issues identified

**Missing values:**
- Customer ID: 243,007 missing (22.8%) — significant for RFM analysis
- Description: 4,382 missing (0.4%) — minor

**Type / format issues:**
- InvoiceDate loaded as `object` — fixed with `parse_dates`
- Customer ID stored as float64 due to NaN presence — cosmetic, will handle in cleaning
- StockCode "M" vs "m" — inconsistent capitalization (1,421 vs 5 rows)
- Descriptions: 5,698 unique for 5,305 StockCodes — same product with varying descriptions

**Business-logic anomalies:**
- **Returns identified via two methods:**
  - 'C' invoice prefix (19,494 rows) — verified as credit notes via £-1.5M revenue impact
  - Negative Quantity without 'C' prefix (3,457 rows) — "hidden" returns / manual reversals
  - Total returns: 22,950 rows (2.15% of dataset)
- **Extreme outlier:** paired 80,995-unit transaction (Invoice 581483 / C581484) — legitimate bulk order + reversal 12 minutes apart
- **Non-product StockCodes** (5,620 rows across 12 codes): POST, DOT, M, C2, D, S, PADS, CRUK, B, m, GIFT, C3 — represent shipping, adjustments, discounts, samples, charity, services

## Decisions carried into cleaning phase (Notebook 02)
1. Add `revenue = Quantity × Price` column
2. Add `is_return = (Quantity < 0) OR (Invoice starts with 'C')`
3. Add `transaction_type` classifier column (Product / Shipping / Adjustment / Discount / Sample / Charity / Service)
4. Normalise "M"/"m" StockCode capitalisation
5. Document handling of 243k missing Customer IDs (exclude from RFM; keep for gross revenue)
6. Add date-derived columns (year, month, day-of-week) for time-series analysis
7. Filter dynamically — do not blanket-delete non-product rows
